## Problem 1. (FFBS - Forward Filtering Backward Sampling) 
Here we discuss situation with discrete spaces 

### Introduction 
Our goal is to generate random hidden states from distribution:
$$p(X_0, \dots, X_k \mid Y_0, \dots, Y_k)$$

### Algorithm 
1. We derive single hidden state distributions $$\alpha_i(x_i) \propto p(x_i|y_0, \ldots, y_i)$$ 
2. We go backwards from $i=k-1$ to $i=0$  


### Methodology 

In [ ]:
import numpy as np
from utils_algorithms_PF_and_FFBS import (
    generate_hmm_data,
    forward_filtering_backward_sampling,
    particle_filter_sis_pfs,
    analysis_plot_state_distributions_with_errors,
    analysis_plot_convergence_rate
)


In [ ]:
# Params
## Define HMM dimensions and pre-defined transition matrices
num_states = 3
sim_length = 100
num_particles = 200


initial_dist = np.array([0.34, 0.33, 0.33])
## Uniform weights array passed as parameter
initial_weights = np.ones(num_particles)  

transition_matrix = np.array([
    [0.8, 0.1, 0.1],
    [0.1, 0.8, 0.1],
    [0.2, 0.1, 0.7]
])

emission_matrix = np.array([
    [0.7, 0.3],
    [0.2, 0.8],
    [0.5, 0.5]
])

## Generate observation vector sequence
true_states, observations = generate_hmm_data(
    initial_dist, transition_matrix, emission_matrix, sim_length
)

## Obtain Ground Truth analytical filtering distribution (num_samples=0)
exact_filtering_dist = forward_filtering_backward_sampling(
    initial_dist, transition_matrix, emission_matrix, observations, num_samples=0
)

In [ ]:
# Analysis 
## params
n_test_list = [10] + [50 * i for i in range(1, 21)]


## Evaluate the convergence velocity of models across various particle capacities
analysis_plot_convergence_rate(initial_dist, transition_matrix, emission_matrix, observations, n_test_list)

In [ ]:
# Analysis

## params
### Slices for evaluation along the time trajectory
test_slices = [1, 5, 15, 30, 45, 60, 75, 99]


## Simulate SIS and PF using identical properties
_, sis_empirical_dist = particle_filter_sis_pfs(
    initial_dist, initial_weights, transition_matrix, emission_matrix, observations,
    num_particles=num_particles, use_resampling=False
)

_, pf_empirical_dist = particle_filter_sis_pfs(
    initial_dist, initial_weights, transition_matrix, emission_matrix, observations,
    num_particles=num_particles, use_resampling=True
)



## Render probability breakdown and performance metrics simultaneously
analysis_plot_state_distributions_with_errors(test_slices, exact_filtering_dist, sis_empirical_dist, pf_empirical_dist)


In [ ]:
import numpy as np

def generate_biological_cpg_data(sim_length: int) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Setup a biological HMM case study for CpG Islands detection and generate testing data.
    
    States:
      0: CpG Island (High G/C content)
      1: Normal DNA (Low G/C content, High A/T)
      
    Emissions:
      0: 'A', 1: 'C', 2: 'G', 3: 'T'
    """
    # 2 states: 0 = CpG Island, 1 = Normal DNA
    initial_dist = np.array([0.1, 0.9])  # Genoms start mostly as normal DNA
    
    # Transition matrix T (rows sum to 1)
    # High probability of staying in the same genomic region type
    transition_matrix = np.array([
        [0.95, 0.05],  # State 0 -> stays 0.95, goes to normal 0.05
        [0.02, 0.98]   # State 1 -> goes to island 0.02, stays normal 0.98
    ])
    
    # Emission matrix L (rows sum to 1)
    # State 0 (CpG): C and G are highly enriched
    # State 1 (Normal): A and T dominate, C and G are suppressed
    emission_matrix = np.array([
        [0.15, 0.35, 0.35, 0.15],  # CpG Island: [A: 15%, C: 35%, G: 35%, T: 15%]
        [0.30, 0.20, 0.20, 0.30]   # Normal DNA: [A: 30%, C: 20%, G: 20%, T: 30%]
    ])
    
    # Use your previously verified generate_hmm_data to create sequences
    from utils_algorithms_PF_and_FFBS import generate_hmm_data
    true_states, observations = generate_hmm_data(
        initial_dist, transition_matrix, emission_matrix, sim_length
    )
    
    metadata = {
        'initial_dist': initial_dist,
        'transition_matrix': transition_matrix,
        'emission_matrix': emission_matrix,
        'mapping_states': {0: 'CpG Island', 1: 'Normal DNA'},
        'mapping_emissions': {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
    }
    
    return true_states, observations, metadata

In [ ]:
# 1. Generate real-case biological data (length 60 bases)
sim_length = 60
num_particles = 100
true_states, observations, meta = generate_biological_cpg_data(sim_length)

# Print a piece of generated sequence to verify biology
dna_string = "".join([meta['mapping_emissions'][o] for o in observations])
print(f"Generated DNA sequence:\n{dna_string}\n")

# Setup default configurations
initial_weights = np.ones(num_particles)

# 2. Extract Ground Truth distribution (num_samples=0)
exact_filtering_dist = forward_filtering_backward_sampling(
    meta['initial_dist'], meta['transition_matrix'], meta['emission_matrix'], observations, num_samples=0
)

# 3. Process SIS & PF on DNA data
_, sis_empirical_dist = particle_filter_sis_pfs(
    meta['initial_dist'], initial_weights, meta['transition_matrix'], meta['emission_matrix'], observations,
    num_particles=num_particles, use_resampling=False
)

_, pf_empirical_dist = particle_filter_sis_pfs(
    meta['initial_dist'], initial_weights, meta['transition_matrix'], meta['emission_matrix'], observations,
    num_particles=num_particles, use_resampling=True
)

# 4. Define time checkpoints to verify (e.g., slices during the genomic scan)
test_slices = [2, 10, 25, 40, 58]

# 5. Draw the beautiful pattern stacked bars with exact error overhead labels
analysis_plot_state_distributions_with_errors(
    test_slices, exact_filtering_dist, sis_empirical_dist, pf_empirical_dist
)

In [ ]:
# 1. Generate real-case biological data (length 60 bases)
sim_length = 60
num_particles = 100
true_states, observations, meta = generate_biological_cpg_data(sim_length)

# Print a piece of generated sequence to verify biology
dna_string = "".join([meta['mapping_emissions'][o] for o in observations])
print(f"Generated DNA sequence:\n{dna_string}\n")

# Setup default configurations
initial_weights = np.ones(num_particles)

# 2. Extract Ground Truth distribution (num_samples=0)
exact_filtering_dist = forward_filtering_backward_sampling(
    meta['initial_dist'], meta['transition_matrix'], meta['emission_matrix'], observations, num_samples=0
)

# 3. Process SIS & PF on DNA data
_, sis_empirical_dist = particle_filter_sis_pfs(
    meta['initial_dist'], initial_weights, meta['transition_matrix'], meta['emission_matrix'], observations,
    num_particles=num_particles, use_resampling=False
)

_, pf_empirical_dist = particle_filter_sis_pfs(
    meta['initial_dist'], initial_weights, meta['transition_matrix'], meta['emission_matrix'], observations,
    num_particles=num_particles, use_resampling=True
)

# 4. Define time checkpoints to verify (e.g., slices during the genomic scan)
test_slices = [2, 10, 25, 40, 58]

# 5. Draw the beautiful pattern stacked bars with exact error overhead labels
analysis_plot_state_distributions_with_errors(
    test_slices, exact_filtering_dist, sis_empirical_dist, pf_empirical_dist
)